# Notebook Joris - pipeline artiste production

Ce notebook garde uniquement le pipeline actuel utile: extraction MusicBrainz, preprocessing, entraînement KNN, sauvegarde des modèles test et suivi des métriques.

Les anciennes cellules de baseline locale ont été retirées pour éviter les erreurs liées aux fichiers `raw_data`, aux variables intermédiaires et aux anciennes versions du modèle.


## Pipeline production artiste: entraînement et prédiction

Cette section regroupe le pipeline actuel utilisé par le projet pour entraîner le modèle artiste et faire une prédiction.

Flux entraînement:

1. `app.database.get_connection()` ouvre la connexion PostgreSQL MusicBrainz.
2. `ml.data.fetch_artist_training_data()` ou `fetch_artist_recommender_training_data()` exécute les requêtes SQL et construit un dataframe avec une ligne par artiste.
3. `ml.features.join_genre_feature_tokens()` transforme les genres en tokens stables: `east coast hip hop` devient `east_coast_hip_hop`.
4. `ml.train.build_artist_recommender_artifact()` entraîne le `TfidfVectorizer` puis le KNN cosine.
5. `ml.artifact.save_artifact()` sauvegarde le `.pkl` localement.

Flux prédiction:

1. `app.predictor.load_latest_model()` charge le dernier artefact local ou GCS.
2. `app.predictor.predict_playlist()` transforme les artistes d'entrée en vecteurs et récupère les voisins KNN.
3. `app.predictor.predict_artist()` recharge les détails artistes en SQL. La requête de sortie place `artist_tag.count >= 1` dans le `LEFT JOIN artist_tag`: les tags faibles sont filtrés sans supprimer les artistes recommandés.


In [ ]:
# Setup notebook -> projet rec_o
# Cette cellule permet d'importer app/ et ml/ depuis notebooks/ ou models/.

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {"notebooks", "models"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


In [ ]:
# Imports du pipeline actuel

from app.database import get_connection
from ml.data import fetch_artist_training_data, fetch_artist_recommender_training_data
from ml.train import build_artist_recommender_artifact, train_artist_recommender
from ml.artifact import save_artifact
from ml.gcs_upload import upload_model_to_gcs
from app.predictor import load_latest_model, predict_playlist, predict_artist

print("Imports OK")


### 1. Récupération SQL des données d'entraînement

Deux modes existent:

- `fetch_artist_training_data(...)`: pratique pour tester avec `max_artists`, `skip_extended_genres` et le cache.
- `fetch_artist_recommender_training_data(...)`: pipeline complet avec tables temporaires SQL, plus proche du vrai run complet.

Pour itérer vite, commencer avec `max_artists=5000` et `use_cache=True`. Pour le modèle final, enlever la limite ou utiliser le pipeline complet.


In [ ]:
# Chargement des données d'entraînement pour le notebook.
# Par défaut, on évite une requête SQL longue: on charge le cache si présent,
# sinon on récupère le dataframe depuis un modèle local existant.

import joblib
import pandas as pd
from pathlib import Path

RUN_SQL_FETCH = False

cache_candidates = [
    PROJECT_ROOT / "ml" / "outputs" / "training_features.pkl",
    PROJECT_ROOT / "ml" / "outputs" / "training_features_min_count_2.pkl",
]
model_candidates = [
    PROJECT_ROOT / "models" / "knn_model_test_joris_slim.pkl",
    PROJECT_ROOT / "models" / "knn_model_test_joris.pkl",
    PROJECT_ROOT / "models" / "knn_baseline_model.pkl",
]

raw_df = None

for candidate in cache_candidates:
    if candidate.is_file():
        raw_df = pd.read_pickle(candidate)
        print(f"Loaded training cache: {candidate}")
        break

if raw_df is None:
    for candidate in model_candidates:
        if candidate.is_file():
            artifact_from_disk = joblib.load(candidate)
            raw_df = artifact_from_disk.get("data")
            if raw_df is None:
                raw_df = artifact_from_disk.get("df_clean")
            if raw_df is not None:
                raw_df = raw_df.copy()
                print(f"Loaded artifact dataframe: {candidate}")
                break

if raw_df is None and RUN_SQL_FETCH:
    with get_connection() as conn:
        raw_df = fetch_artist_training_data(
            conn,
            max_artists=5000,
            skip_extended_genres=False,
            use_cache=True,
            refresh_cache=False,
        )

if raw_df is None:
    raise FileNotFoundError(
        "Aucun cache/model local trouvé. Mettre RUN_SQL_FETCH=True pour lancer la requête SQL."
    )

print(raw_df.shape)
raw_df[["artist_id", "artist_name", "genres"]].head()


### 1 bis. Contrôles preprocessing

Avant d'entraîner le modèle, on vérifie les points qui peuvent créer du bruit ou casser la correspondance KNN -> artiste:

- valeurs manquantes (`NaN`)
- genres vides ou à zéro information
- doublons complets
- doublons sur `artist_id`
- tags avec `tag_count_sum` nul quand la colonne est disponible


In [ ]:
# Contrôles preprocessing avant entraînement

required_columns = ["artist_id", "artist_name", "genres"]
missing_columns = [col for col in required_columns if col not in raw_df.columns]
if missing_columns:
    raise ValueError(f"Colonnes manquantes dans raw_df: {missing_columns}")

preprocessing_report = {
    "rows": len(raw_df),
    "nan_by_column": raw_df.isna().sum().to_dict(),
    "full_duplicates": int(raw_df.duplicated().sum()),
    "artist_id_duplicates": int(raw_df.duplicated(subset=["artist_id"]).sum()),
    "empty_or_nan_genres": int(raw_df["genres"].fillna("").astype(str).str.strip().eq("").sum()),
}

if "tag_count_sum" in raw_df.columns:
    preprocessing_report["tag_count_sum_nan"] = int(raw_df["tag_count_sum"].isna().sum())
    preprocessing_report["tag_count_sum_zero_or_less"] = int(raw_df["tag_count_sum"].fillna(0).le(0).sum())

preprocessing_report


In [ ]:
# Nettoyage défensif avant entraînement.
# build_artist_recommender_artifact() refiltre aussi les genres vides,
# mais cette cellule rend le preprocessing visible dans le notebook.

training_df = raw_df.copy()
training_df["artist_name"] = training_df["artist_name"].fillna("")
training_df["genres"] = training_df["genres"].fillna("").astype(str).str.split().str.join(" ")

training_df = training_df.drop_duplicates()
training_df = training_df.drop_duplicates(subset=["artist_id"])
training_df = training_df[training_df["genres"].str.strip() != ""].copy()

if "tag_count_sum" in training_df.columns:
    training_df["tag_count_sum"] = training_df["tag_count_sum"].fillna(0).clip(lower=0)

print(f"Rows before preprocessing: {len(raw_df):,}")
print(f"Rows after preprocessing: {len(training_df):,}")
training_df[["artist_id", "artist_name", "genres"]].head()


In [ ]:
# Version complète, plus lente: tables temporaires SQL sur la base complète.
# A utiliser pour produire un modèle final.

# with get_connection() as conn:
#     raw_df = fetch_artist_recommender_training_data(
#         conn,
#         use_cache=True,
#         refresh_cache=False,
#     )
#
# print(raw_df.shape)
# raw_df[["artist_id", "artist_name", "genres"]].head()


### 2. Entraînement du modèle

Le modèle est un KNN cosine entraîné sur des vecteurs TF-IDF de genres.

L'artefact sauvegardé contient:

```python
{
    "vectorizer": vectorizer,
    "model": knn_model,
    "artist_names": artist_names,
    "data": df_clean,
    "genre_feature_format": "genre_token_unigram",
}
```


In [ ]:
artifact = build_artist_recommender_artifact(training_df, n_neighbors=20)

print(type(artifact["vectorizer"]).__name__)
print(type(artifact["model"]).__name__)
print(artifact["genre_feature_format"])
print(artifact["data"].shape)


In [ ]:
# Sauvegarde locale du modèle.
# Ecrit une copie timestampée dans models/, une copie canonique et une copie dans ml/outputs/.

model_path = save_artifact(artifact)
model_path


In [ ]:
# Upload GCS optionnel après validation locale.
# Nécessite MODEL_BUCKET_NAME / MODEL_BLOB_NAME et des credentials GCP valides.

# upload_model_to_gcs(model_path)


### 2 bis. Modèle test Joris

Cette cellule sauvegarde un artefact séparé pour comparer les performances sans écraser le modèle principal.

Nom local: `knn_model_test_joris.pkl`

Blob GCS de test: `models/knn_model_test_joris.pkl`


In [ ]:
# Sauvegarde + upload du modèle test Joris
# Cette version n'utilise pas save_artifact(), car save_artifact met aussi à jour
# la copie canonique. Ici on garde volontairement un fichier de test séparé.

from pathlib import Path
import joblib
from google.cloud import storage
from ml.config import MODEL_BUCKET_NAME, MODEL_DIR, ML_OUTPUTS_DIR

TEST_MODEL_FILENAME = "knn_model_test_joris.pkl"
TEST_MODEL_BLOB_NAME = "models/knn_model_test_joris.pkl"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
ML_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

test_model_path = MODEL_DIR / TEST_MODEL_FILENAME
test_output_path = ML_OUTPUTS_DIR / TEST_MODEL_FILENAME

joblib.dump(artifact, test_model_path)
joblib.dump(artifact, test_output_path)

print(f"Saved test model: {test_model_path}")
print(f"Saved test output copy: {test_output_path}")

# Upload GCS optionnel, vers un blob de test séparé.
# Décommenter après validation locale et credentials GCP OK.
# if not MODEL_BUCKET_NAME:
#     raise RuntimeError("MODEL_BUCKET_NAME is not set in .env")
# client = storage.Client()
# bucket = client.bucket(MODEL_BUCKET_NAME)
# blob = bucket.blob(TEST_MODEL_BLOB_NAME)
# blob.upload_from_filename(test_model_path)
# print(f"Uploaded to gs://{MODEL_BUCKET_NAME}/{TEST_MODEL_BLOB_NAME}")

test_model_path


### 2 quater. Run min_tag_count=2

Le pipeline accepte maintenant `min_tag_count`. Pour tester un modèle moins bruité, on peut régénérer les features avec `min_tag_count=2`.

Ce run écrit un cache séparé: `ml/outputs/training_features_min_count_2.pkl`, donc il ne remplace pas le cache `>=1`.

Résultat observé sur le run complet:

- `data_rows`: 324606 au lieu de 930628
- `vocab_size`: 1238 au lieu de 1828
- modèle slim: 36.96 MB
- latence moyenne: 68.769 ms

C'est beaucoup plus léger. À valider qualitativement sur des artistes connus, car la couverture baisse fortement.


In [ ]:
# Exemple reproductible pour relancer le modèle test avec seuil >= 2.

# with get_connection() as conn:
#     raw_df_min2 = fetch_artist_recommender_training_data(
#         conn,
#         use_cache=True,
#         refresh_cache=False,
#         min_tag_count=2,
#     )
#
# artifact_min2 = build_artist_recommender_artifact(raw_df_min2, n_neighbors=20)
# artifact_min2_slim = artifact_min2.copy()
# artifact_min2_slim["data"] = artifact_min2_slim["data"][["artist_id", "artist_name", "genres"]].copy()
#
# min2_model_path = PROJECT_ROOT / "models" / "knn_model_test_joris_min_count_2_slim.pkl"
# joblib.dump(artifact_min2_slim, min2_model_path)
# min2_model_path


### 2 quinquies. Run pondéré par source

Test pondéré ajouté au tableau de métriques. La colonne `genres` répète les tokens selon la source:

- artiste/direct: poids 4
- release group: poids 3
- release: poids 2
- recording fallback: poids 1

Résultat observé pour `knn_model_test_joris_weighted_slim_001`:

- couverture conservée: 930628 artistes
- modèle plus lourd: 187.97 MB
- latence moyenne: 200.74 ms
- diversité des genres recommandés: 141 genres uniques sur le smoke test
- `genre_jaccard_mean` plus bas: 0.5037, signe que les recommandations sont moins collées aux mêmes genres exacts

À valider qualitativement: le modèle est plus expressif, mais il peut introduire plus de bruit et l'artefact grossit.


In [ ]:
# Exemple reproductible pour relancer le modèle pondéré.

# from ml.data import fetch_weighted_artist_recommender_training_data
#
# with get_connection() as conn:
#     raw_df_weighted = fetch_weighted_artist_recommender_training_data(
#         conn,
#         use_cache=True,
#         refresh_cache=False,
#         min_tag_count=1,
#     )
#
# artifact_weighted = build_artist_recommender_artifact(raw_df_weighted, n_neighbors=20)
# artifact_weighted_slim = artifact_weighted.copy()
# artifact_weighted_slim["data"] = artifact_weighted_slim["data"][["artist_id", "artist_name", "genres"]].copy()
#
# weighted_model_path = PROJECT_ROOT / "models" / "knn_model_test_joris_weighted_slim.pkl"
# joblib.dump(artifact_weighted_slim, weighted_model_path)
# weighted_model_path


### 2 ter. Métriques comparatives

Le fichier `ml/artist_model_metrics.csv` sert de tableau de suivi pour comparer les essais.

Colonnes importantes pour décider si un modèle est meilleur:

- `data_rows`, `vocab_size`, `genre_tokens_mean`, `genre_tokens_p95`: complexité et bruit des features.
- `model_size_mb`, `load_seconds`, `prediction_latency_mean_ms`: coût runtime.
- `empty_genres`, `duplicate_artist_ids`, `nan_total`: qualité preprocessing.
- `genre_jaccard_mean`, `recommended_unique_genres`, `seed_leak_count`, `recommendation_duplicate_count`: qualité simple des recommandations.

Pour les prochains essais, relancer `python -m ml.scripts.collect_artist_model_metrics --model-path models/knn_model_test_joris.pkl --run-id features_count_ge_1_release_tags_001 --notes "description du test"`.


In [ ]:
import pandas as pd
from pathlib import Path

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name in {"notebooks", "models"}:
        PROJECT_ROOT = PROJECT_ROOT.parent

metrics_path = PROJECT_ROOT / "ml" / "artist_model_metrics.csv"
metrics_df = pd.read_csv(metrics_path)

metrics_df[[
    "run_id",
    "data_rows",
    "vocab_size",
    "model_size_mb",
    "load_seconds",
    "prediction_latency_mean_ms",
    "genre_tokens_mean",
    "genre_tokens_p95",
    "empty_genres",
    "duplicate_artist_ids",
    "genre_jaccard_mean",
    "recommended_unique_genres",
    "notes",
]]


### 3. Prédiction directe en Python

Cette partie reproduit ce que fait l'endpoint FastAPI `/predict/artist` après la validation de la clé API.

`predict_playlist()` renvoie des `artist_id`. Ensuite `predict_artist()` recharge les détails SQL pour obtenir `gid`, `name`, `genre` et `urls`.


In [ ]:
seed_artist_ids = [347307]  # Remplacer par des MusicBrainz artist.id présents dans la base
top_n = 5

load_latest_model()

with get_connection() as conn:
    recommended_ids = predict_playlist(seed_artist_ids, top_n=top_n)
    recommended_artists = predict_artist(recommended_ids, conn)

recommended_ids, recommended_artists[["gid", "name", "genre", "urls"]]


### 4. Requête API équivalente

Quand l'API tourne avec `uvicorn app.main:app --reload`, le client envoie le même type d'entrée à `/predict/artist`.


In [ ]:
import os
import requests

api_url = "http://localhost:8000/predict/artist"
headers = {"X-API-Key": os.getenv("TOKEN_API_KEY", "<token>")}
payload = {"ArtistIds": seed_artist_ids, "TopN": top_n}

# response = requests.post(api_url, json=payload, headers=headers, timeout=30)
# response.raise_for_status()
# response.json()


### 5. Point important sur les genres renvoyés

Dans `app.predictor.predict_artist()`, la requête SQL de sortie filtre les tags artiste dans le `LEFT JOIN` avec:

```sql
AND artist_tag.count >= 1
```

Cela évite de renvoyer des genres basés sur des tags trop faibles (`count` inférieur à 1).


### Ouverture du tableau de métriques

Dernière cellule pratique pour consulter le CSV de comparaison des modèles depuis le notebook.


In [ ]:
# Ouverture finale du CSV de métriques

import pandas as pd
from pathlib import Path

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name in {"notebooks", "models"}:
        PROJECT_ROOT = PROJECT_ROOT.parent

metrics_path = PROJECT_ROOT / "ml" / "artist_model_metrics.csv"
metrics_df = pd.read_csv(metrics_path)

metrics_df
